In [11]:
# Cell 1 - import wing-x
import pandas as pd
import numpy as np

# 1. SETUP & LOAD
METRO_PATH = "gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro.parquet"
print("🚀 Loading and processing 230-pair market audit...")
df = pd.read_parquet(METRO_PATH)

# 2. STANDARDIZE & FILTER
df['FlightDate_ET'] = pd.to_datetime(df['FlightDate_ET'])
df = df.rename(columns={'FromCluster': 'FromMetro', 'ToCluster': 'ToMetro'})

# --- CRITICAL FIX: Filter FIRST, then define corridor on the clean data ---
mask = (
    (df['aircraft_segment'] == 'Super Midsize Jet') & 
    (df['FromMetro'] != 'OTHER_METRO') & 
    (df['ToMetro'] != 'OTHER_METRO') & 
    (df['FromMetro'] != df['ToMetro']) & # This eliminates Intra-Metro
    (df['Hours'] >= 2.5)
)

# Create smid_lh as a clean, independent copy
smid_lh = df[mask].copy()

# Now define the corridor only for the legitimate inter-metro pairs
smid_lh['corridor'] = smid_lh['FromMetro'] + " ➔ " + smid_lh['ToMetro']

# 3. WUP PRESENCE DEFINITION
wup_mask = ((smid_lh['Operator'].str.contains('Wheels Up', case=False, na=False)) | 
            (smid_lh['Operator'] == 'Mountain Aviation'))
smid_lh['is_wup'] = wup_mask

# 4. BI-DIRECTIONAL AGGREGATION
def make_pair_key(row):
    return " <-> ".join(sorted([row['FromMetro'], row['ToMetro']]))

smid_lh['pair_key'] = smid_lh.apply(make_pair_key, axis=1)

# Aggregation logic
pair_analysis = smid_lh.groupby('pair_key').agg(
    total_pair_missions=('pair_key', 'size'),
    total_wup_missions=('is_wup', 'sum'),
    avg_hours_combined=('Hours', 'mean')
).reset_index()

def get_splits(group):
    counts = group['corridor'].value_counts()
    hrs = group.groupby('corridor')['Hours'].mean()
    wup = group.groupby('corridor')['is_wup'].sum()
    total = group.groupby('corridor').size()
    return pd.Series({
        'mission_split': " | ".join(counts.astype(str)),
        'hours_split': " | ".join([f"{h:.2f}" for h in hrs]),
        'share_split': " | ".join([f"{(w/t)*100:.1f}%" for w, t in zip(wup, total)])
    })

# 5. VALIDATION & OUTPUT
splits = smid_lh.groupby('pair_key').apply(get_splits).reset_index()
pair_analysis = pair_analysis.merge(splits, on='pair_key').sort_values('total_pair_missions', ascending=False)

# Validation check to ensure it's clean
intra_metro_count = smid_lh[smid_lh['FromMetro'] == smid_lh['ToMetro']].shape[0]
print(f"✅ Market Audit Complete.")
print(f"1. Total Inter-Metro Missions: {len(smid_lh):,}")
print(f"2. Unique Directional Corridors: {smid_lh['corridor'].nunique()}")
print(f"3. Intra-Metro (Same City) Missions: {intra_metro_count}")

display(pair_analysis.head(10))

🚀 Loading and processing 230-pair market audit...
✅ Market Audit Complete.
1. Total Inter-Metro Missions: 211,133
2. Unique Directional Corridors: 447
3. Intra-Metro (Same City) Missions: 0


/var/tmp/ipykernel_15563/3728518009.py:59: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  splits = smid_lh.groupby('pair_key').apply(get_splits).reset_index()


,pair_key,total_pair_missions,total_wup_missions,avg_hours_combined,mission_split,hours_split,share_split
200,New York City <-> South Florida,13756,130,2.728246,8054 | 5702,2.76 | 2.68,1.2% | 0.6%
130,Denver <-> New York City,8041,460,3.821391,4032 | 4009,3.50 | 4.14,5.9% | 5.6%
57,Boston <-> South Florida,7900,257,2.895832,4268 | 3632,3.02 | 2.76,3.5% | 2.9%
92,Chicago <-> South Florida,7087,121,2.736125,4212 | 2875,2.67 | 2.78,1.6% | 1.8%
137,Denver <-> South Florida,6377,478,3.935630,3269 | 3108,3.64 | 4.25,7.3% | 7.7%
166,LA Basin <-> New York City,5939,104,4.962665,3066 | 2873,4.60 | 5.35,2.0% | 1.5%
82,Chicago <-> LA Basin,5113,172,3.561338,2596 | 2517,3.83 | 3.28,3.2% | 3.6%
173,LA Basin <-> South Florida,4716,159,4.647834,2424 | 2292,4.21 | 5.11,3.2% | 3.5%
113,Dallas <-> LA Basin,4637,72,2.802512,3305 | 1332,2.85 | 2.68,1.8% | 1.1%
116,Dallas <-> New York City,4351,90,3.062016,2289 | 2062,2.87 | 3.23,1.8% | 2.3%


In [53]:
# -----------------------------------------------------------------------------------------
# CELL 2: DIRECTIONAL SEASONAL DISCOVERY — SMID
# -----------------------------------------------------------------------------------------

from sklearn.cluster import KMeans

# 1. Build monthly corridor fingerprint
seasonal_counts = (
    smid_lh
    .groupby(["corridor", "month"])
    .size()
    .unstack(fill_value=0)
)

# Ensure all month columns exist
seasonal_counts = seasonal_counts.reindex(columns=range(1, 13), fill_value=0)

# 2. Filter for statistical signal
MIN_MISSIONS = 30

annual_missions = seasonal_counts.sum(axis=1)
relevant_corridors = annual_missions > MIN_MISSIONS

seasonal_counts_relevant = seasonal_counts.loc[relevant_corridors].copy()

# 3. Normalize to monthly density
# This isolates the seasonal shape of each corridor.
monthly_density = seasonal_counts_relevant.div(
    seasonal_counts_relevant.sum(axis=1),
    axis=0
)

# 4. Create seasonality index
# 1.00x = normal monthly demand
# >1.00x = above-normal seasonal intensity
# <1.00x = below-normal seasonal intensity
seasonality_index = monthly_density / (1 / 12)

# 5. K-Means clustering
km = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=50
)

cluster_ids = km.fit_predict(monthly_density)

# 6. Build audit/output dataframe
X_discovery = monthly_density.copy()
X_discovery["cluster_id"] = cluster_ids
X_discovery["annual_missions"] = annual_missions.loc[relevant_corridors] / 2

for m in range(1, 13):
    X_discovery[f"seasonality_index_m{m}"] = seasonality_index[m]

# 7. Display cluster centers — monthly density
cluster_centers_density = (
    X_discovery
    .groupby("cluster_id")[list(range(1, 13))]
    .mean()
    .reindex(columns=range(1, 13))
)

print(f"✅ Processed {len(X_discovery)} SMID corridors into 4 seasonal demand regimes.")
print("\n📈 MONTHLY DENSITY BY CLUSTER:")
display(
    cluster_centers_density
    .style
    .background_gradient(axis=1, cmap="YlOrRd")
    .format("{:.1%}")
)

# 8. Display cluster centers — seasonality index
seasonality_cols = [f"seasonality_index_m{m}" for m in range(1, 13)]

cluster_centers_index = (
    X_discovery
    .groupby("cluster_id")[seasonality_cols]
    .mean()
)

cluster_centers_index.columns = range(1, 13)

print("\n📈 SEASONALITY INDEX BY CLUSTER:")
display(
    cluster_centers_index
    .style
    .background_gradient(axis=1, cmap="YlOrRd")
    .format("{:.2f}x")
)

✅ Processed 289 SMID corridors into 4 seasonal demand regimes.

📈 MONTHLY DENSITY BY CLUSTER:


month,1,2,3,4,5,6,7,8,9,10,11,12
cluster_id,,,,,,,,,,,,
0,12.7%,11.9%,12.1%,10.6%,8.8%,4.3%,3.5%,3.7%,4.5%,6.7%,10.2%,11.0%
1,4.1%,4.1%,5.2%,6.6%,9.1%,11.3%,12.4%,12.6%,10.7%,10.5%,7.0%,6.3%
2,7.1%,7.6%,7.9%,8.7%,9.4%,7.9%,6.9%,7.2%,8.7%,10.6%,9.6%,8.5%
3,9.2%,9.3%,10.0%,4.9%,5.4%,8.8%,11.5%,10.8%,8.4%,6.6%,5.1%,9.9%



📈 SEASONALITY INDEX BY CLUSTER:


,1,2,3,4,5,6,7,8,9,10,11,12
cluster_id,,,,,,,,,,,,
0,1.52x,1.43x,1.45x,1.27x,1.05x,0.51x,0.42x,0.45x,0.54x,0.80x,1.23x,1.32x
1,0.49x,0.50x,0.63x,0.80x,1.09x,1.36x,1.49x,1.51x,1.29x,1.26x,0.84x,0.75x
2,0.85x,0.91x,0.95x,1.04x,1.13x,0.94x,0.83x,0.86x,1.04x,1.27x,1.16x,1.02x
3,1.11x,1.11x,1.20x,0.59x,0.65x,1.06x,1.38x,1.30x,1.00x,0.80x,0.62x,1.19x


In [54]:
# -----------------------------------------------------------------------------------------
# CELL 3.5: LABELED SEASONALITY AUDIT (SYNCED)
# -----------------------------------------------------------------------------------------

# 1. Re-define labels locally to force a sync (Matches your Cell 4 logic)
final_labels = {
    0: "winter-spring-peak",   
    1: "summer-peak",        
    2: "multi-peak-directional", 
    3: "core-utilization"
}

# 2. Calculate centers from X_discovery
labeled_centers = X_discovery.groupby("cluster_id")[list(range(1, 13))].mean()

# 3. Apply labels to the index (Prepend ID for clarity)
labeled_centers.index = [f"{i}: {final_labels.get(i, 'Unknown')}" for i in labeled_centers.index]

print("🎯 FINAL SMID ARCHETYPE HEATMAP (Monthly Density %):")
display(
    labeled_centers.style.background_gradient(axis=1, cmap="YlOrRd")
    .format("{:.1%}")
)

# 4. Toggle Logic Calculation
# May is index 5, June is index 6
summer_jump = labeled_centers[6] - labeled_centers[5]
top_toggle = summer_jump.idxmax()

print(f"\n💡 The strongest June 'Toggle' candidate is: {top_toggle}")
print(f"📈 Momentum: +{summer_jump.max():.1%} absolute density increase MoM.")

🎯 FINAL SMID ARCHETYPE HEATMAP (Monthly Density %):


month,1,2,3,4,5,6,7,8,9,10,11,12
0: winter-spring-peak,12.7%,11.9%,12.1%,10.6%,8.8%,4.3%,3.5%,3.7%,4.5%,6.7%,10.2%,11.0%
1: summer-peak,4.1%,4.1%,5.2%,6.6%,9.1%,11.3%,12.4%,12.6%,10.7%,10.5%,7.0%,6.3%
2: multi-peak-directional,7.1%,7.6%,7.9%,8.7%,9.4%,7.9%,6.9%,7.2%,8.7%,10.6%,9.6%,8.5%
3: core-utilization,9.2%,9.3%,10.0%,4.9%,5.4%,8.8%,11.5%,10.8%,8.4%,6.6%,5.1%,9.9%



💡 The strongest June 'Toggle' candidate is: 3: core-utilization
📈 Momentum: +3.4% absolute density increase MoM.


In [56]:
# -----------------------------------------------------------------------------------------
# FINAL VALIDATION: SMID UNIVERSE & VOLUME DISTRIBUTION
# -----------------------------------------------------------------------------------------

# 1. Labels strictly aligned with Cell 3.5 Heatmap
final_labels_smid = {
    0: "winter-spring-peak",   
    1: "summer-peak",        
    2: "multi-peak-directional", 
    3: "core-utilization"
}

# 2. Metadata & Volume Calculations
total_corridors = len(X_discovery)
total_flights = X_discovery['annual_missions'].sum()

# Grouping by mapped labels for volume analysis
stats = X_discovery.groupby('cluster_id').agg({'annual_missions': ['count', 'sum']})
stats.index = stats.index.map(final_labels_smid)
stats.columns = ['corridor_count', 'flight_volume']
stats['volume_pct'] = (stats['flight_volume'] / total_flights) * 100

# 3. Intra-Metro Check
intra_metro_issues = [c for c in X_discovery.index if c.split(' ➔ ')[0] == c.split(' ➔ ')[1]]

# 4. Validation Output
print("📊 --- SMID CLUSTER UNIVERSE SUMMARY ---")
print(f"📌 Filtering Criteria Applied:")
print(f"   - Segment      : Super Midsize Jet")
print(f"   - Min Leg Time : >= 2.5 Hours")
print(f"   - Relevance    : > {MIN_MISSIONS} missions total (2-yr)")
print(f"   - Logic        : Inter-Metro Only (No Same-City)")

print(f"\n✅ Total Inter-Metro Corridors : {total_corridors}")
print(f"✅ Total Annual Flight Volume  : {total_flights:,.0f} (Annualized /2)")

print("\n📍 SMID ARCHETYPE BREAKDOWN (Volume & Count):")
print(f"{'Archetype':<25} | {'Corridors':<10} | {'Flights':<12} | {'Volume %'}")
print("-" * 65)
for label, row in stats.sort_values('flight_volume', ascending=False).iterrows():
    print(f"{label:<25} | {int(row['corridor_count']):<10} | {int(row['flight_volume']):<12,} | {row['volume_pct']:.1f}%")

print("\n🛡️ --- SAFETY CHECKS ---")
if not intra_metro_issues:
    print(f"✅ Final Validation Passed: 0 intra-metro corridors found.")
else:
    print(f"❌ Final Validation Failed: Found {len(intra_metro_issues)} intra-metro records!")

📊 --- SMID CLUSTER UNIVERSE SUMMARY ---
📌 Filtering Criteria Applied:
   - Segment      : Super Midsize Jet
   - Min Leg Time : >= 2.5 Hours
   - Relevance    : > 30 missions total (2-yr)
   - Logic        : Inter-Metro Only (No Same-City)

✅ Total Inter-Metro Corridors : 289
✅ Total Annual Flight Volume  : 104,992 (Annualized /2)

📍 SMID ARCHETYPE BREAKDOWN (Volume & Count):
Archetype                 | Corridors  | Flights      | Volume %
-----------------------------------------------------------------
multi-peak-directional    | 117        | 44,008       | 41.9%
winter-spring-peak        | 61         | 28,404       | 27.1%
core-utilization          | 41         | 18,716       | 17.8%
summer-peak               | 70         | 13,863       | 13.2%

🛡️ --- SAFETY CHECKS ---
✅ Final Validation Passed: 0 intra-metro corridors found.
